# Занятие 5. Очистка, фильтрация и data card маленького корпуса

**Цель практики:** собрать мини-корпус через Wikipedia API, удалить дубли/короткие тексты, посчитать простые признаки качества и создать черновик data card.

Работает в бесплатном Colab на CPU.

In [ ]:
import os, re, json, textwrap, math, statistics, random, io, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

## 1. Скачиваем мини-корпус

In [ ]:
LANG = 'udm'
API = f'https://{LANG}.wikipedia.org/w/api.php'

def random_extracts(n=25):
    params = {
        'action': 'query',
        'generator': 'random',
        'grnnamespace': 0,
        'grnlimit': n,
        'prop': 'extracts|info',
        'explaintext': 1,
        'inprop': 'url',
        'format': 'json',
    }
    r = requests.get(API, params=params, timeout=30)
    r.raise_for_status()
    pages = r.json().get('query', {}).get('pages', {})
    return pd.DataFrame([{
        'title': p.get('title'),
        'url': p.get('fullurl'),
        'text': p.get('extract', ''),
    } for p in pages.values()])

df = random_extracts(25)
df.loc[len(df)] = df.iloc[0]  # искусственный дубль для практики
show_df(df[['title', 'url']], 10)

## 2. Чистим текст и считаем признаки

In [ ]:
def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

clean = df.copy()
clean['text_clean'] = clean['text'].apply(normalize_space)
clean['chars'] = clean['text_clean'].str.len()
clean['tokens'] = clean['text_clean'].apply(lambda x: len(re.findall(r'\w+', x)))
clean['cyrillic_share'] = clean['text_clean'].apply(lambda x: len(re.findall(r'[А-Яа-яЁёӐ-ӿ]', x)) / max(len(x), 1))
clean['duplicate'] = clean.duplicated('text_clean')
clean['keep'] = (clean['chars'] >= 300) & (clean['cyrillic_share'] >= 0.45) & (~clean['duplicate'])

show_df(clean[['title', 'chars', 'tokens', 'cyrillic_share', 'duplicate', 'keep']], 30)
filtered = clean[clean['keep']].copy()
save_artifact('lesson05_clean_corpus.csv', filtered[['title', 'url', 'text_clean', 'chars', 'tokens', 'cyrillic_share']])

## 3. Делим на train/dev/test без обучения модели

In [ ]:
filtered = filtered.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(filtered)
filtered['split'] = 'train'
filtered.loc[filtered.index >= int(n * 0.8), 'split'] = 'dev'
filtered.loc[filtered.index >= int(n * 0.9), 'split'] = 'test'
show_df(filtered[['title', 'chars', 'split']], 30)
save_artifact('lesson05_splits.csv', filtered[['title', 'url', 'split', 'text_clean']])

## 4. Черновик data card

In [ ]:
data_card = f'''# Data card: mini {LANG} Wikipedia corpus

## Source
Wikipedia API, language edition: {LANG}.wikipedia.org

## Size
- Raw documents: {len(df)}
- Kept documents: {len(filtered)}
- Total kept characters: {int(filtered['chars'].sum()) if len(filtered) else 0}

## Filtering
- Removed exact duplicates
- Removed documents shorter than 300 characters
- Required Cyrillic-script share >= 0.45

## Known limitations
- Wikipedia is not representative of all language varieties
- Articles may contain named entities, Russian borrowings, lists and formatting remnants
- Human language review is still required

## License
Check Wikipedia/Wikimedia terms and page histories before redistribution.
'''
print(data_card)
save_artifact('lesson05_DATA_CARD.md', data_card)

## Вопросы для отчёта

1. Что фильтры удалили ошибочно?
2. Какие признаки качества не видны из автоматических метрик?
3. Что обязательно должен проверить носитель/эксперт?
4. Что нужно добавить в data card перед публикацией?